In [17]:
import PF_wrapper as PF
import numpy as np
import pandas as pd
import random

import pickle

#from aeon.datasets import load_japanese_vowels
from aeon.datasets import load_arrow_head
#from aeon.classification.distance_based import KNeighborsTimeSeriesClassifier
#from aeon.classification.shapelet_based import RDSTClassifier
from aeon.classification.convolution_based import MiniRocketClassifier

In [12]:
#classifier = KNeighborsTimeSeriesClassifier(n_neighbors=9)

In [26]:
X3, y3 = load_arrow_head(split="TRAIN", return_type="numpy2d")
X3t, y3t = load_arrow_head(split="TEST", return_type="numpy2d")

In [27]:
model = MiniRocketClassifier()
model.fit(X3,y3)
model.score(X3t,y3t)

0.8628571428571429

In [30]:
int(model.predict([X3[0]])[0])

0

In [31]:
with open("minirocket_arrowhead.pkl", "wb") as f:
    pickle.dump(model, f)

In [32]:
def inject_missing_values(data, missing_rate=0.1, seed=None):
    """
    Inject missing values into a dataset.

    Parameters:
    - data: numpy array (2D or 3D) or list of 2D numpy arrays
    - missing_rate: proportion of values to set as missing
    - seed: random seed for reproducibility

    Returns:
    - data with missing values injected (same type as input)
    """
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    def inject_into_array(arr):
        arr = np.copy(arr)
        total_elements = arr.size
        num_missing = int(total_elements * missing_rate)
        indices = [(i, j) for i in range(arr.shape[0]) for j in range(arr.shape[1])]
        missing_indices = random.sample(indices, num_missing)
        for i, j in missing_indices:
            arr[i][j] = np.nan
        return arr

    if isinstance(data, list):
        return [inject_into_array(arr) for arr in data]
    elif isinstance(data, np.ndarray):
        if data.ndim == 2:
            return inject_into_array(data)
        elif data.ndim == 3:
            return np.array([inject_into_array(data[i]) for i in range(data.shape[0])])
        else:
            raise ValueError("Unsupported array dimensionality: expected 2D or 3D numpy array")
    else:
        raise TypeError("Unsupported data type: expected numpy array or list of numpy arrays")

In [43]:
X3df = pd.DataFrame(X3)
X3df.to_csv('Data/arrowhead.txt', index=False, header=None)

X3tdf = pd.DataFrame(inject_missing_values(X3t, missing_rate=0.1, seed=42))
X3tdf.to_csv('Data/arrowhead_test.txt', index=False, header=None)

In [34]:
y3df = pd.DataFrame(y3)
y3df.to_csv("Data/arrowheadlabels.csv", index=False, header=None)

y3tdf = pd.DataFrame(y3t)
y3tdf.to_csv("Data/arrowheadlabels_test.csv", index=False, header=None)

In [35]:
dir1 = "training_output"
dir2 = "training_predictions"

In [39]:
# First, we train a PF model and give it the name 'Spartacus'

PF.train("Data/arrowhead.txt", train_labels="Data/arrowheadlabels.csv",
                  output_directory=dir1, array_separator=":", entry_separator=",", 
                  model_name="meta_arrowhead", data_dimension=1, num_trees=11, parallel_trees=False,
        distances=["meta_classmatch:Rocket.py:predict:class"])


0:3mb
finished in 0:0:0.013

-----------------Repetition No: 1 (arrowhead.txt)   -----------------
Using: 3 MB, Free: 21 MB, Allocated Pool: 24 MB, Max Available: 1024 MB
core.ProximityForestResult@7cd84586
0.1.2.3.4.5.6.7.8.9.10.
Using: 10 MB, Free: 14 MB, Allocated Pool: 24 MB, Max Available: 1024 MB


In [45]:
# Now let's get predictions on the test set.
PF.predict(dir1 + "/meta_arrowhead", "Data/arrowhead_test.txt", test_labels="Data/arrowheadlabels_test.csv",
           entry_separator=",", array_separator=":", data_dimension=1, output_directory=dir2,
          impute_testing_data=True, initial_imputer="linear", return_imputed_testing=True,
          distances=["meta_classmatch:Rocket.py:predict:class"], return_predictions=True)


0:7mb
finished in 0:0:0.038
Performing initial imputation...
Updating missing values...
Updating missing values...
Updating missing values...
Updating missing values...
Updating missing values...
**


In [46]:
# Here are the predictions (of the saved model) on the training set.
f0 = open(dir2 + "/Predictions_saved.txt")
f1 = f0.read()
imputed_preds_saved = eval("np.array(" + f1 + ")")
f0.close()

In [48]:
imputed_preds_saved

array([0, 2, 0, 2, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 2, 0, 0, 0, 1, 0,
       0, 0, 1, 0, 0, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 1,
       0, 2, 0, 2, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

In [52]:
rights = [1 for i in range(len(y3t)) if imputed_preds_saved[i]==int(y3t[i])]
len(rights)/len(y3t)

0.8685714285714285

In [53]:
#0.8628571428571429

In [15]:
# Here are the predictions (of the saved model) on the test set.
f0 = open("Predictions_saved.txt")
f1 = f0.read()
preds_saved = eval("np.array(" + f1 + ")")
f0.close()

In [16]:
print(len(train_preds_saved))
print(len(preds_saved))
print(len(preds))

270
370
370


In [17]:
# Just checking: are the outputs of the saved model equal to the original predictions?
np.unique([preds[i]-preds_saved[i] for i in range(len(preds))])

array([0])

In [18]:
# the following can be used to obtain the training proximities
p=PF.getArray(dir1 + "/TrainingProximities.txt")

In [19]:
p.shape

(270, 270)

In [20]:
# We can also access the test/train proximities
pt=PF.getArray(dir1 + "/TestTrainProximities.txt")

In [21]:
pt.shape

(370, 270)

In [22]:
# The raw proximities are not symmetric. But in some applications, one desires symmetry.
p = 0.5*(p + p.transpose())

In [23]:
# The following can be used to obtain outlier scores for the training set.
# Note that these are intra-class outlier scores.
outlier_scores = PF.getArray(dir1 + "/outlier_scores.txt")
outlier_scores.shape

(270,)